# AI Suite Python SDK demo

Ready-to-use notebook for driving the Arango AI Suite from Python with `arango-ai-sdk`. You should not have to copy gateway URLs, auth headers, or JSON bodies. The only file you edit is `env`: credentials and API keys.

It runs the SDK from sign-in through a question:

1. Sign in
2. Create / get a project
3. Store secret profiles
4. Upload the documents with `upload_folder`
5. Deploy AutoGraph and wait until it is serving
6. Build the corpus, choose strategies, read them, then import the graph
7. Deploy a retriever and ask it a question

`ingest` is that same work as one call. It is shown after the steps and not run. Run the steps or that call, not both: a second build of the same categories raises `CategoryAlreadyBuiltError`.

**Not in this notebook:** the project overview, and deleting projects, secrets, or files.

## How to run it

1. Once, outside the notebook: install the `gcloud` CLI, run `gcloud auth login` and `gcloud auth application-default login` with an account that can read the `arango-ml` project. The SDK is on the Arango private package index and the install cells read it with those credentials.
2. Copy `env.example` to `env` next to this notebook and fill in the empty values: gateway, ArangoDB login, a unique `PROJECT_NAME`, providers, models and API keys. The install settings at the top are already set. Do not change the notebook cells.
3. Select a Python 3.12+ kernel, run the **preflight** cell, then the two **install** cells.
4. Run top to bottom. `./corpus/tech_articles` holds 50 Wikipedia articles about technology companies, products, and people, the same corpus as the Arango Contextual Data Platform tutorials in the documentation. The folder name is the category label.

> **Use a fresh `PROJECT_NAME` for each run.** The SDK cannot delete a project yet, so re-using a name fails at step 2.

> `env` holds real credentials. Keep it out of git and off shared screens.


---
## Preflight

Check the kernel and the files before installing anything. The SDK needs Python 3.12 or newer; some notebook images still ship 3.11.

In [ ]:
import sys
from pathlib import Path

HERE = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "AI_Suite_SDK_demo.ipynb").is_file()),
    None,
)
if HERE is None:
    raise SystemExit("Start Jupyter in the folder that holds AI_Suite_SDK_demo.ipynb.")
env_path = HERE / "env"
corpus_path = HERE / "corpus"
if not env_path.is_file():
    raise SystemExit("Copy env.example to env next to the notebook and fill it in.")
if not corpus_path.is_dir():
    raise SystemExit("The folder 'corpus' is missing next to the notebook.")

print("python    :", sys.version)
print("3.12+     :", sys.version_info >= (3, 12))
print("folder    :", HERE)
print("env       :", "found")
print("categories:", sorted(p.name for p in corpus_path.iterdir() if p.is_dir()))

## Install

Two cells. The first installs the helpers from public PyPI: `python-dotenv` to read `env`, and the keyring backend that lets `pip` sign in to Google Artifact Registry with your `gcloud` login. The second reads `env`, where the index URL and the SDK version live, and installs the SDK from the private index.

If the second cell cannot find the package, the `gcloud` login from step 1 is missing or the account cannot read the `arango-ml` project. If you re-run it to upgrade the SDK, restart the kernel afterwards.

In [ ]:
%pip install --no-input python-dotenv keyring keyrings.google-artifactregistry-auth

In [ ]:
import os

from dotenv import load_dotenv


def setting(name: str) -> str:
    """Read one value from env, re-reading the file so edits apply without a restart."""
    load_dotenv(env_path, override=True)
    return os.environ[name]


SDK_VERSION = setting("SDK_VERSION")
%pip install "arango-ai-sdk==$SDK_VERSION"

---
## 0. Configuration

`env` holds connection info and keys only. Never print keys.

Every value is read through `setting()`, which re-reads `env` on each call. If you change the file, for example a new `PROJECT_NAME` for another run, re-run this cell and the new values apply; no kernel restart needed.

In [ ]:
import arango_ai_sdk

GATEWAY = setting("GATEWAY").rstrip("/")
USERNAME = setting("ARANGO_USERNAME")
PASSWORD = setting("ARANGO_PASSWORD")
DB = setting("DB_NAME")
PROJECT = setting("PROJECT_NAME")
CHAT_API_KEY = setting("CHAT_API_KEY")
EMBEDDING_API_KEY = setting("EMBEDDING_API_KEY") or CHAT_API_KEY
CHAT_PROVIDER = setting("CHAT_API_PROVIDER")
EMBEDDING_PROVIDER = setting("EMBEDDING_API_PROVIDER")
CHAT_MODEL = setting("CHAT_MODEL")
EMBEDDING_MODEL = setting("EMBEDDING_MODEL")
FPS_RECOVERY_USERNAME = setting("FPS_RECOVERY_USERNAME")
CORPUS_DIR = (HERE / setting("CORPUS_DIR")).resolve()

missing = [
    name
    for name, value in {
        "GATEWAY": GATEWAY,
        "ARANGO_USERNAME": USERNAME,
        "ARANGO_PASSWORD": PASSWORD,
        "DB_NAME": DB,
        "PROJECT_NAME": PROJECT,
        "CHAT_API_KEY": CHAT_API_KEY,
        "CHAT_API_PROVIDER": CHAT_PROVIDER,
        "EMBEDDING_API_PROVIDER": EMBEDDING_PROVIDER,
        "CHAT_MODEL": CHAT_MODEL,
        "EMBEDDING_MODEL": EMBEDDING_MODEL,
        "FPS_RECOVERY_USERNAME": FPS_RECOVERY_USERNAME,
    }.items()
    if not value
]
if missing:
    raise SystemExit(f"Fill in {', '.join(missing)} in env.")

print("sdk     :", arango_ai_sdk.__version__)
print("gateway :", GATEWAY)
print("db      :", DB)
print("project :", PROJECT)
print("corpus  :", CORPUS_DIR)
print("models  :", CHAT_MODEL, "/", EMBEDDING_MODEL)

---
## The same flow without the SDK

Until now, driving the suite from Python meant talking to the gateway yourself over HTTP with a JWT. The snippets below are what that costs you. They are here for reference only — **nothing in this section runs**.

You start as a client of the gateway rather than a user of a library, so you own the plumbing:

```python
# Reference only — not executed
S = requests.Session()

class ApiError(RuntimeError):
    def __init__(self, method, path, status_code, body):
        super().__init__(f"{method} {path} -> {status_code}: {body[:600]}")
        self.status_code = status_code

def authenticate():
    r = S.post(f"{GATEWAY}/_db/_system/_open/auth",
               json={"username": USERNAME, "password": PASSWORD})
    r.raise_for_status()
    S.headers.update({"Authorization": f"Bearer {r.json()['jwt']}"})

def api(method, path, *, base=None, **kw):
    url = (base or GATEWAY) + path
    r = S.request(method, url, **kw)
    if not r.ok:
        raise ApiError(method, path, r.status_code, r.text)
    return r.json() if r.content else {}
```

**The JWT lasts about 60 minutes,** so you re-authenticate before anything slow. Forget it and a long upload dies with a 401 halfway through.

**You have to know the internal layout.** A wrong prefix is a silent 404:

- `POST /_db/_system/_open/auth`
- `POST /_platform/acp/v1/project`
- `GET /_platform/acp/v1/project_by_name/{db}/{project}`
- `POST /_platform/acp/v1/secrets`
- File Manager `POST .../rag-input/batch` (multipart)
- AutoGraph `{AG}/v1/corpus/builds`, `/rag-strategizer/analyze`, `/orchestrate`

**The wire format leaks into your code.** Responses mix camelCase and snake_case (`category` vs the older `module`), so you write a reader for both:

```python
# Reference only — not executed
def _camel(name: str):
    return re.sub(r"_(\w)", lambda m: m.group(1).upper(), name)

def pick(obj, *names, default=None):
    for name in names:
        for key in (name, _camel(name)):
            if isinstance(obj, dict) and key in obj:
                return obj[key]
    return default
```

The upload scope has to be `[project, category]`, or the corpus build later cannot resolve the files.

**Every long-running job is a polling loop you write yourself.** After deploy, the three ingestion jobs are each a POST, then a GET you repeat until the status is terminal. A 404 while the pod is still coming up is "not ready", not a failed job. You also decide what a status string means, and you catch one HTTP error and branch on the code:

```python
# Reference only — not executed
# POST {service}/v1/corpus/builds          then GET .../builds/{id}
# POST {service}/v1/rag-strategizer/analyze then GET .../jobs/{id}
# GET  {service}/v1/rag-strategizer/strategy   # the graph-vs-vectors choice
# POST {service}/v1/orchestrate            then GET .../orchestrate/{id}

def wait_until(poll_fn, is_done, desc="", every=10, timeout=7200):
    ...
```

The strategizer job status only has progress counts. The choice for each cluster, full graph or vectors, is a second read. If the graph is already up to date, orchestrate answers 409 "nothing to orchestrate". That is success, not a failure you should raise.

**Uploading is a multipart body, and deploying is an install `env` map** — not "upload a folder" and "start AutoGraph":

```python
# Reference only — not executed
def upload_batch(files_by_category: dict[str, list[Path]]):
    # files parts + scope fields, path.read_bytes()
    ...

api("POST", "/_platform/acp/v1/service/...", json={"env": {
    "GENAI_PROJECT_NAME": PROJECT,
    "CHAT_SECRET_PROFILE_ID": chat_id,
    "FPS_RECOVERY_USERNAME": FPS_USER,
    # providers, models, ...
}})
```

**Errors are a status code and a body.** You catch one generic error and branch on `e.status_code` to work out what actually went wrong.

The rest of this notebook does the same five steps as method calls.

---
## The same flow with the SDK

| Step | Over HTTP | With the SDK |
| --- | --- | --- |
| Sign in | `authenticate()` + Bearer header on every call | `ArangoAIClient(url, user, password, db_name)` |
| Project | `POST /_platform/acp/v1/project`, then read the JSON | `client.project.create` returns a project handle |
| Secrets | `POST /_platform/acp/v1/secrets` | `client.secret_profile.create` / `.get` |
| Upload | Hand-built multipart (`/rag-input/batch`) | `client.files.upload_files(...)` or `client.files.upload_folder(...)` |
| Deploy | Install `env` map | `project.autograph.deploy` / `.undeploy` |
| Model config | `PUT {AG}/v1/projects/{project}/model-config/credentials` | sent with `project.autograph.deploy` |
| Corpus build | `POST {AG}/v1/corpus/builds` + poll | `project.autograph.corpus_build(...)` waits and returns the final status |
| Strategizer | `POST {AG}/v1/rag-strategizer/analyze` + poll | `project.autograph.strategize(...)` waits; the job status has progress counts only |
| Stored strategies | `GET {AG}/v1/rag-strategizer/strategy` | `project.autograph.strategies()` reads the graph-versus-vectors choice |
| Orchestrate | `POST {AG}/v1/orchestrate` + poll | `project.autograph.orchestrate(...)` waits, or returns `None` when nothing is stale |
| One call | no single HTTP endpoint; you post and poll the three jobs yourself | `project.autograph.ingest(...)` runs them and returns every outcome |
| Pod health | poll `GET {AG}/v1/health` yourself | `project.autograph.wait_until_ready()` |
| Ask | Retriever HTTP (`LOCAL` / `GLOBAL` / `UNIFIED`) | `project.autorag.deploy`, then `retriever.ask(...)` |

The rows through the question run in this notebook. `ingest` is the same work as sections 6 through 9. It is shown below and not run, because those steps already built these categories.

Install the package, pass your credentials once, call named methods.

---
## 1. Sign in

Constructing the client authenticates. The token and its renewal live inside the SDK, so nothing below sets a header. Bad credentials fail here with `InvalidCredentialsError`.


In [ ]:
from arango_ai_sdk import ArangoAIClient

client = ArangoAIClient(
    base_url=GATEWAY,
    username=USERNAME,
    password=PASSWORD,
    db_name=DB,
)
print("signed in as:", client.username)
print("database    :", client.db_name)

---
## 2. Create the project

`create` returns a project handle, an object with `.name`, `.file_names`, and services such as `.autograph` — instead of a JSON dict you have to read key by key. `get` fetches an existing one.

Use a fresh `PROJECT_NAME` in `env` so re-runs do not collide.

In [ ]:
project = client.project.create(
    project_name=PROJECT,
    project_type="autograph",
    description="Created from the AI Suite SDK demo notebook.",
)
fetched = client.project.get(project_name=PROJECT)
print("created:", project.name, project.project_type, project.db_name)
print("fetched:", fetched.name, fetched.project_type)

---
## 3. Store LLM credentials as secret profiles

The raw API keys go to the platform once. Every later step refers to them by `profile_id`, so the key never travels through your code again. Print the ids, never the keys.

In [ ]:
chat = client.secret_profile.create(
    name=f"{PROJECT}-chat",
    secret=CHAT_API_KEY,
    provider=CHAT_PROVIDER,
)
embed = client.secret_profile.create(
    name=f"{PROJECT}-embed",
    secret=EMBEDDING_API_KEY,
    provider=EMBEDDING_PROVIDER,
)
print("chat :", chat.profile_id)
print("embed:", embed.profile_id)
print("get  :", client.secret_profile.get(chat.profile_id).name)

---
## 4. Upload files

Documents live under `scope = [project, category]`. Here the category is `tech_articles`, the name of the folder.

**`upload_folder`** — files sitting *directly* in one folder, all under one scope. Subfolders are ignored. Used here for `corpus/tech_articles/`.

**`upload_files`** — several files in one request: a list that shares one `scope`, or a mapping of path → scope when categories differ. One request is at most 100 files or 2 GiB. There is also `upload_file` for a single path.

Do not call `upload_folder(CORPUS_DIR)` on the corpus root — that would skip every article because they live in a subfolder.

Each result should come back `status == "ok"`. The cell prints the count and any file that failed. Listing, downloading, and deleting files are not in the SDK yet.

In [ ]:
# upload_folder: every file directly inside one category folder
articles_dir = CORPUS_DIR / "tech_articles"
results = client.files.upload_folder(articles_dir, scope=[PROJECT, articles_dir.name])
failed = [r for r in results if r.status != "ok"]
print("upload_folder", articles_dir.name)
print("  uploaded:", len(results) - len(failed), "of", len(results))
for r in failed:
    print(" ", r.name, r.status, r.detail)

---
## 5. Deploy AutoGraph

The same step as configuring and installing the service in the UI, with the install parameters as keyword arguments.

Success here means the platform accepted the install. `deploy` returns a `DeployResult`: `service_info.service_id` is set, and `ready` is `None` because this cell does not pass `wait=True`. `is_deployed` becomes `True`. `service_url` is an in-cluster address, not something you open in a browser.

The next line waits until the pod answers its health check. That wait defaults to 300 seconds. It is only "is the process up", not a limit on the build or the import. If it returns `False`, the pod did not become ready in time and the cells below would talk to a service that is not there yet.

Re-running deploy on a project that already has AutoGraph raises `AutoGraphAlreadyDeployedError`.

`fps_recovery_username` must already exist with read/write on `DB_NAME`. The SDK does not create that user.

In [ ]:
result = project.autograph.deploy(
    chat_api_provider=CHAT_PROVIDER,
    embedding_api_provider=EMBEDDING_PROVIDER,
    chat_secret_profile_id=chat.profile_id,
    embedding_secret_profile_id=embed.profile_id,
    fps_recovery_username=FPS_RECOVERY_USERNAME,
    chat_model=CHAT_MODEL,
    embedding_model=EMBEDDING_MODEL,
)
print("service_id :", result.service_info.service_id)
print("ready      :", result.ready)
print("is_deployed:", project.autograph.is_deployed)
print("service_url:", project.autograph.service_url)

serving = project.autograph.wait_until_ready()
print("serving    :", serving)
if not serving:
    raise SystemExit("AutoGraph did not become ready within 300 seconds.")

---
## 6. Build the corpus

The category label is the folder name, `tech_articles`. `corpus_build` starts the build and waits until it finishes. You do not write the poll loop.

`timeout` is optional. Leave it unset and this cell waits until the build finishes. Passing `timeout=` would cap this step only. It is not the 300 second pod check above.

A completed build with `error_code` set is a partial success. The call returns it. It does not raise.

In [ ]:
CATEGORIES = ["tech_articles"]


def show_build(status):
    print("build", status.status, status.progress)


# timeout= would cap this step. Unset means wait until the build finishes.
build = project.autograph.corpus_build(CATEGORIES, on_progress=show_build)
print("status     :", build.status)
print("documents  :", build.document_count)
print("error_code :", build.error_code or "")

---
## 7. Choose a strategy per cluster

`strategize` decides, for each cluster, whether it is a full graph or vectors, and waits until that job finishes. The default complexity is `high`. The status this returns only has progress counts, such as how many clusters are done.

In [ ]:
def show_strategize(status):
    print("strategize", status.status, f"{status.clusters_done}/{status.clusters_total}")


strategize_status = project.autograph.strategize(CATEGORIES, on_progress=show_strategize)
print("status        :", strategize_status.status)
print("clusters_done :", strategize_status.clusters_done)

---
## 8. Read the stored strategies

The job status does not say which cluster got a graph and which got vectors. `strategies()` reads that stored result. It does not start another job.

In [ ]:
stored = project.autograph.strategies()
print("total :", stored.total_strategies)
print("counts:", dict(stored.strategy_type_counts))

---
## 9. Import the graph

`orchestrate` imports the knowledge graph and waits. If every category is already up to date it returns `None`. That is success. There is nothing to catch.

In [ ]:
def show_orchestrate(status):
    print("orchestrate", status.status, status.phase)


orchestration = project.autograph.orchestrate(CATEGORIES, on_progress=show_orchestrate)
if orchestration is None:
    print("nothing was stale")
else:
    print("status:", orchestration.status)
    print("phase :", orchestration.phase)

---
## 10. The one call

`ingest` runs the corpus build, the strategizer, the strategy read, and the import, then returns all four outcomes. There is no single HTTP endpoint for that. `timeout` on this call is one budget for the whole run, not a fresh one per step. `orchestration` on the result is `None` when nothing was stale.

Run sections 6 through 9, or run the next cell, not both. Those sections already build these categories, and a second build raises `CategoryAlreadyBuiltError`.

Leave the next cell commented when you run sections 6 through 9. To use this call instead, skip sections 6 through 9, uncomment the cell, and run it. It sets `CATEGORIES` itself, because that name is otherwise created in section 6.

In [ ]:
# Same work as sections 6 through 9.
# Leave this commented when you run those sections.
# Skip sections 6 through 9, uncomment the lines below, and run this cell instead.
#
# CATEGORIES = ["tech_articles"]
#
# def show_ingest(progress):
#     print(progress.step, progress.status.status)
#
# result = project.autograph.ingest(CATEGORIES, on_progress=show_ingest)
# print("build        :", result.build.status)
# print("strategize   :", result.strategize.status)
# print("strategies   :", result.strategies.total_strategies)
# print("orchestration:", None if result.orchestration is None else result.orchestration.status)

---
## 11. Deploy a retriever

A project can have more than one retriever. `project.autorag` is the manager: `deploy` installs a new one, `list` and `get` find the ones already there.

The arguments are the same model settings as AutoGraph, without `fps_recovery_username`. Success means the platform accepted the install. `deploy` returns an `AutoRagDeployResult`: `service_info.service_id` is set, `ready` is `None` because this cell does not pass `wait=True`, and `autorag` is the object bound to that service.

The next line waits until the retriever answers its health check with `OK`. That wait defaults to 300 seconds. It is only "is the process up", not a limit on the question. If it returns `False`, the pod did not become ready in time.


In [ ]:
retriever = project.autorag.deploy(
    chat_api_provider=CHAT_PROVIDER,
    embedding_api_provider=EMBEDDING_PROVIDER,
    chat_secret_profile_id=chat.profile_id,
    embedding_secret_profile_id=embed.profile_id,
    chat_model=CHAT_MODEL,
    embedding_model=EMBEDDING_MODEL,
)
print("service_id :", retriever.service_info.service_id)
print("ready      :", retriever.ready)
rag = retriever.autorag
print("is_deployed:", rag.is_deployed)
print("service_url:", rag.service_url)

serving = rag.wait_until_ready()
print("serving    :", serving)
if not serving:
    raise SystemExit("The retriever did not become ready within 300 seconds.")

---
## 12. Ask a question

`ask` sends one question and returns the answer. `query_type` defaults to `QueryType.UNIFIED`, which works on both a full graph and vectors. `GLOBAL` fails when a partition is vectors only. The response has `result` and `run_id`.


In [ ]:
from arango_ai_sdk.models import QueryType

answer = rag.ask(
    "Which companies has Microsoft acquired, and when?",
    query_type=QueryType.UNIFIED,
)
print("run_id:", answer.run_id)
print(answer.result)

---
## 13. List and look up

`list` returns every retriever on the project. `get` returns the one with that service id. A service id that is not there raises `AutoRagNotDeployedError`.


In [ ]:
listed = project.autorag.list()
print("retrievers:", [item.service_id for item in listed])
same = project.autorag.get(rag.service_id)
print("got:", same.service_id, "is_deployed:", same.is_deployed)

---
## Failures have names

Over HTTP you catch one error and branch on its status code. The SDK raises specific exceptions — `InvalidCredentialsError`, `InvalidProjectNameError`, `AutoGraphAlreadyDeployedError` — all under `ArangoAIError` if you want to catch everything at once.

A corpus build, strategizer run, or import that ends as failed raises `IngestionFailedError`. The exception carries `job_status`, the last status that step returned. A corpus build that finishes with a partial problem does not raise: `corpus_build` returns, and `error_code` on that status says what was partial. Building a category that is already built raises `CategoryAlreadyBuiltError`. This notebook builds each category once, so it does not hit these cases.

Asking a retriever that is not in the project raises `AutoRagNotDeployedError`. A question the retriever answers with an error code raises `QueryFailedError`. That exception carries `failure_code` and `run_id`.


The next cell signs in with a wrong password. It creates and changes nothing on the platform.

In [ ]:
from arango_ai_sdk.exceptions import InvalidCredentialsError

try:
    ArangoAIClient(GATEWAY, USERNAME, "wrong-password", db_name=DB)
except InvalidCredentialsError as e:
    print(type(e).__name__, "— caught by name, no status code to inspect")

---
## What comes next

Still missing (out of scope for V1): the project overview, listing and deleting files, and deleting projects, secrets, and categories.


---
## Clean up

Remove the retriever, then the AutoGraph service, then close the client. The SDK cannot delete projects, secret profiles, or files yet.


In [ ]:
removed_rag = rag.undeploy()
print("retriever undeployed:", removed_rag.service_id, "is_deployed:", rag.is_deployed)
removed = project.autograph.undeploy()
print("autograph undeployed:", removed.service_id, "is_deployed:", project.autograph.is_deployed)
client.close()
print("client closed")